# Видимый фрагмент при сильном перекрытии

Подключите данные конкурса и приватный clevr_rim. Выберите GPU T4 ×2. Обучение использует train, разметка val нужна для диагностики.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("rim_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_rim.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_rim.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/rim_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

In [ ]:
assert all("T4" in torch.cuda.get_device_name(i) for i in range(2)), "Выберите GPU T4 ×2"
env.update(FRAGMENT_INPUT=str(package), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", NUMEXPR_NUM_THREADS="2")

RESULT = ROOT/"rim"
RESULT.mkdir(exist_ok=True)
ARMS = ("rgb","edge","unet","r50","r50native","r50real","r18","convnext","efficient")
SEEDS = (421,853)
EPOCHS=24
SCENE_EPOCHS=24
V="scripts.research.rim_fragments"
U="scripts.research.rim_scene_unet"
R="scripts.research.rear_candidate"
E="scripts.research.rim_eval"
from threading import Lock
failure_lock=Lock()
failures=[]
def launch(gpu, module, *args):
    step = module.rsplit(".",1)[-1]+"_"+"_".join(map(str,args)).replace("--","")
    done = RESULT / f"{step}.done"
    if done.exists():
        return
    log = RESULT / f"{step}.log"
    cmd = [sys.executable, "-m", module, *map(str,args)]
    with log.open("a", buffering=1) as out, subprocess.Popen(cmd, cwd=ROOT, text=True,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as job:
        for line in job.stdout:
            print(f"[GPU {gpu}] {line}", end="", flush=True)
            out.write(line)
        if job.wait():
            shutil.copy2(log, WORK/f"rim_{step}_failed.log")
            raise RuntimeError(f"{step} завершился с ошибкой. Лог сохранён в Output.")
    done.touch()

def parallel(jobs):
    with ThreadPoolExecutor(max_workers=2) as pool:
        results = [pool.submit(fn,i,*args) for i,(fn,args) in enumerate(jobs)]
        for result in results:
            result.result()

def monitor(job, stage):
    telemetry = RESULT / f"gpu_{stage}.csv"
    with telemetry.open("w",buffering=1) as out:
        query = ["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
                 "--format=csv", "-l", "2"]
        try:
            process = subprocess.Popen(query,stdout=out,stderr=subprocess.STDOUT)
        except FileNotFoundError:
            process = None
        try:
            return job()
        finally:
            if process is not None:
                process.terminate()
                process.wait()

def safe(gpu,module,*args):
    try:
        launch(gpu,module,*args)
        return True
    except Exception as exc:
        record=dict(job=module,args=list(args),error=str(exc))
        with failure_lock:
            failures.append(record)
            (RESULT/"failures.json").write_text(json.dumps(failures,indent=2))
        print("Сбой шага",record,flush=True)
        return False

def pair(module, tasks):
    with ThreadPoolExecutor(2) as pool:
        pending=[pool.submit(safe,gpu,module,*args) for gpu,args in enumerate(tasks)]
        return [job.result() for job in pending]

def extract(split):
    return all(pair(V,[("prepare","--split",split,"--shard",k) for k in (0,1)]))

def posterior(split):
    return all(pair(R,[("prepare","--split",split,"--shard",k) for k in (0,1)]))

def infer_local(arm,seed,split):
    return all(pair(V,[("infer","--arm",arm,"--seed",seed,"--split",split,"--shard",k) for k in (0,1)]))

## Локальные модели и самостоятельный U-Net

Каждая локальная модель видит кроп 40×40 в исходных пикселях и проверяет слабый пик R50. Кандидаты и границы передней фигуры задаются одинаково для всех вариантов. U-Net на полном кадре выделяет собственные маски, считает и сравнивает найденные фигуры. Каждый вариант обучается с двумя seed.

In [ ]:
def train():
    prepared=extract("val")
    real_ready=extract("train")
    if prepared:
        for arm in ARMS:
            if arm=="r50real" and not real_ready:
                continue
            fit=pair(V,[("fit","--arm",arm,"--seed",s,"--epochs",EPOCHS) for s in SEEDS])
            for s,ok in zip(SEEDS,fit):
                if ok:
                    infer_local(arm,s,"val")
    fit=pair(U,[("fit","--seed",s,"--epochs",SCENE_EPOCHS) for s in SEEDS])
    if all(fit):
        pair(U,[("val","--seed",s) for s in SEEDS])
    else:
        for gpu,(s,ok) in enumerate(zip(SEEDS,fit)):
            if ok:
                safe(gpu,U,"val","--seed",s)
    posterior("val")

monitor(train,"train")

## Сравнение на исходных сценах

Сравниваем попадания по отмеченным фрагментам, совпадение атрибутов, ложные кандидаты и ROC-AUC поверх count_hidden. Валидируем подписи на всём val. Четыре пользовательские рамки не участвуют в обучении или выборе порога.

In [ ]:
evaluation=safe(0,E)
report=json.loads((RESULT/"metrics.json").read_text()) if evaluation else {}
if report:
    print("ROC-AUC базы",report["base"]["auc"],"R50 count",report["r50_count"]["auc"])
    rows=[]
    diagnostics=[]
    for model,d in report["experiments"].items():
        rows.extend(dict(run=k,**value) for k,value in d["metrics"].items())
        if model.endswith("_both"):
            diagnostics.extend(dict(run=model,threshold=k,**value) for k,value in d["operating"].items())
    if rows:
        display(pd.DataFrame(rows).sort_values("auc",ascending=False).drop(columns="auc_by_category").head(30))
        display(pd.DataFrame(diagnostics))
    scene_rows=[]
    for seed,results in report["scene_unet"].items():
        scene_rows.append(dict(seed=seed,counts_and_comparison_auc=results["core_auc"],
                               replacement_auc=results["replacement"]["auc"],blend_auc=results["blend"]["auc"]))
    if scene_rows:
        display(pd.DataFrame(scene_rows))
print("Вариант для test",report.get("winner"))

## Сохранение результатов

Скачайте rim_results.zip с метриками, весами, логами и прогнозами. Если выбран локальный вариант для test, ноутбук также сохраняет CSV.

In [ ]:
winner=report.get("winner")
def run_test():
    if not winner:
        return
    arm=winner[0]
    if arm=="scene_unet":
        if not all(pair(U,[("test","--seed",s) for s in SEEDS])):
            return
    else:
        if not extract("test") or not posterior("test"):
            return
        for seed in SEEDS:
            if not infer_local(arm,seed,"test"):
                return
    if safe(0,E,"--export"):
        name="unet_scene.csv" if arm=="scene_unet" else f"rim_{arm}.csv"
        shutil.copy2(RESULT/name,WORK/name)
        display(FileLink(name))

monitor(run_test,"test")
with zipfile.ZipFile(WORK/"rim_results.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in RESULT.glob("*"):
        if (path.is_file() and not path.name.endswith("_rgb.npy") and
            path.suffix in {".json",".npz",".pt",".log",".csv"}):
            z.write(path,path.name)
os.chdir(WORK)
display(FileLink("rim_results.zip"))